# 03 · Pemodelan: Regresi vs Klasifikasi

**Langkah [3]–[4] dari alur aplikasi.** Membandingkan dua pendekatan untuk memprediksi **kelas kompensasi PM 89/2015 (0–5)**:

| | Pendekatan A — Regresi | Pendekatan B — Klasifikasi |
|---|---|---|
| Target | menit delay → dikonversi ke kelas 0–5 | kelas 0–5 langsung |
| Skenario | **S1** target asli · **S2** target log | **S3** tanpa class weight · **S4** class weight *balanced* |
| Algoritma | Linear Regression, Decision Tree, Random Forest, XGBoost | Logistic Regression, Decision Tree, Random Forest, XGBoost |

Total **16 model**. Evaluasi: split stratified 80:20 + 5-fold CV.
Metrik utama **macro F1** dan **recall kelas 5**; regresi juga MAE, RMSE, R².

Aman dijalankan ulang: model yang hasilnya sudah tersimpan di Drive akan dilewati.
Perkiraan waktu di Colab gratis: ± 30–60 menit untuk semua model.

## 1. Setup

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/Data BTS 2023-2025'
except ImportError:
    BASE = './data'

import os, json, time
import numpy as np, pandas as pd, duckdb
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, accuracy_score,
                             f1_score, precision_recall_fscore_support, confusion_matrix)
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBRegressor, XGBClassifier

GLOB = os.path.join(BASE, 'parquet', '*', '*.parquet').replace(os.sep, '/')
MODEL_DIR = os.path.join(BASE, 'models')
RESULT_DIR = os.path.join(MODEL_DIR, 'results')
FIG_DIR = os.path.join(BASE, 'figures')
for d in (MODEL_DIR, RESULT_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

SEED = 42
N_SAMPLE = 1_000_000   # sampel dari ±20,6 juta penerbangan; naikkan kalau waktu & RAM cukup
CV_N = 200_000         # subset data latih untuk 5-fold CV (agar RF tetap selesai di Colab gratis)
CLASSES = [0, 1, 2, 3, 4, 5]
BLUE, BLUE2, ORANGE, GOLD = '#1E3A8A', '#3B82F6', '#F97316', '#F5B700'

## 2. Kelas PM 89/2015

Aturan yang **sama persis** dengan `02_eda` (SQL) dan nanti web (TypeScript):
`< 30 → 0`, `30–60 → 1`, `61–120 → 2`, `121–180 → 3`, `181–240 → 4`, `> 240 → 5`.

In [ ]:
def to_class(minutes):
    # shortcut: aturan kelas diduplikasi di 02_eda (SQL) & web; satukan kalau mulai berubah-ubah
    d = np.asarray(minutes, dtype=float)
    return ((d >= 30).astype(int) + (d > 60) + (d > 120) + (d > 180) + (d > 240)).astype(int)

assert list(to_class([0, 29.9, 30, 60, 60.5, 120, 121, 180, 181, 240, 240.5, 999])) == [0, 0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5]

## 3. Ambil sampel

Sampel diambil dengan **hash deterministik** (hasil sama setiap dijalankan), proporsional terhadap seluruh data,
lalu dibagi **stratified 80:20** berdasarkan kelas. Penerbangan batal/divert tidak diikutkan.

In [ ]:
con = duckdb.connect()
n_ops = con.execute(f'''SELECT COUNT(*) FROM read_parquet('{GLOB}')
    WHERE Cancelled = 0 AND Diverted = 0 AND DepDelayMinutes IS NOT NULL''').fetchone()[0]
per10k = max(1, min(10_000, round(10_000 * N_SAMPLE / n_ops)))

df = con.execute(f'''
SELECT Reporting_Airline AS airline, Origin AS origin, Dest AS dest,
       Month AS month, DayOfWeek AS dow, CRSDepTime AS crs_dep,
       Distance AS distance, CRSElapsedTime AS crs_elapsed,
       DepDelayMinutes AS delay
FROM read_parquet('{GLOB}')
WHERE Cancelled = 0 AND Diverted = 0 AND DepDelayMinutes IS NOT NULL
  AND CRSElapsedTime IS NOT NULL AND Distance IS NOT NULL
  AND hash(FlightDate, Reporting_Airline, Flight_Number_Reporting_Airline, Origin) % 10000 < {per10k}
''').df()
df['cls'] = to_class(df.delay)
print(f'{len(df):,} baris dari {n_ops:,} penerbangan beroperasi')
df.cls.value_counts(normalize=True).sort_index().mul(100).round(3)

## 4. Fitur

Hanya informasi yang **sudah diketahui sebelum terbang** (bisa diisi pengguna di web):

| Fitur | Perlakuan |
|---|---|
| maskapai, bandara asal, bandara tujuan | kode kategori (one-hot untuk model linear) |
| bulan, jam jadwal berangkat | siklik sin/cos (Desember dekat Januari, 23:00 dekat 00:00) |
| hari dalam minggu | angka 1–7 (one-hot untuk model linear) |
| jarak, durasi terjadwal | angka (distandarisasi untuk model linear) |

Mapping kategori → kode disimpan ke `models/encoders.json` agar web memakai kode yang identik.

In [ ]:
CAT = ['airline', 'origin', 'dest']
FEATURES = CAT + ['dow', 'month_sin', 'month_cos', 'hour_sin', 'hour_cos', 'distance', 'crs_elapsed']

train_df, test_df = train_test_split(df, test_size=0.2, stratify=df.cls, random_state=SEED)
categories = {c: sorted(train_df[c].astype(str).unique()) for c in CAT}

def make_features(d):
    X = pd.DataFrame(index=d.index)
    for c in CAT:
        X[c] = pd.Categorical(d[c].astype(str), categories=categories[c]).codes  # tidak dikenal → -1
    X['dow'] = d.dow.astype(int)
    m = d.month.astype(float)
    X['month_sin'], X['month_cos'] = np.sin(2 * np.pi * m / 12), np.cos(2 * np.pi * m / 12)
    dep = d.crs_dep.astype(int)
    mins = ((dep // 100) * 60 + dep % 100) % 1440
    X['hour_sin'], X['hour_cos'] = np.sin(2 * np.pi * mins / 1440), np.cos(2 * np.pi * mins / 1440)
    X['distance'] = d.distance.astype(float)
    X['crs_elapsed'] = d.crs_elapsed.astype(float)
    return X[FEATURES].astype('float32')

X_train, X_test = make_features(train_df), make_features(test_df)
y_train_min, y_test_min = train_df.delay.to_numpy(), test_df.delay.to_numpy()
y_train_cls, y_test_cls = train_df.cls.to_numpy(), test_df.cls.to_numpy()

with open(os.path.join(MODEL_DIR, 'encoders.json'), 'w') as f:
    json.dump({'features': FEATURES, 'categories': categories,
               'class_rule': '<30:0, <=60:1, <=120:2, <=180:3, <=240:4, >240:5'}, f)
print('train', X_train.shape, '| test', X_test.shape)

## 5. Definisi 16 model

In [ ]:
def linear_pre():
    return ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), CAT + ['dow']),
        ('num', StandardScaler(), ['month_sin', 'month_cos', 'hour_sin', 'hour_cos', 'distance', 'crs_elapsed']),
    ])

# Hyperparameter tetap (tanpa tuning) agar perbandingan adil & muat di Colab gratis.
DT = dict(max_depth=12, min_samples_leaf=100, random_state=SEED)
RF = dict(n_estimators=100, max_depth=16, min_samples_leaf=50, max_samples=0.5, n_jobs=-1, random_state=SEED)
XGB = dict(n_estimators=400, max_depth=8, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
           tree_method='hist', n_jobs=-1, random_state=SEED)

ALGOS = {
    'regression': {
        'lr':  lambda: make_pipeline(linear_pre(), LinearRegression()),
        'dt':  lambda: DecisionTreeRegressor(**DT),
        'rf':  lambda: RandomForestRegressor(**RF),
        'xgb': lambda: XGBRegressor(**XGB),
    },
    'classification': {
        'lr':  lambda: make_pipeline(linear_pre(), LogisticRegression(max_iter=300)),
        'dt':  lambda: DecisionTreeClassifier(**DT),
        'rf':  lambda: RandomForestClassifier(**RF),
        'xgb': lambda: XGBClassifier(objective='multi:softprob', **XGB),
    },
}
SCENARIOS = {'regression': ['S1', 'S2'], 'classification': ['S3', 'S4']}
SPECS = [(a, s, k) for a in ALGOS for s in SCENARIOS[a] for k in ALGOS[a]]
print(len(SPECS), 'model')

def fit_predict(approach, scenario, algo, X, y_min, y_cls, X_eval):
    """Latih satu model; kembalikan (model, prediksi kelas, prediksi menit atau None)."""
    model = ALGOS[approach][algo]()
    if approach == 'regression':
        target = np.log1p(y_min) if scenario == 'S2' else y_min
        model.fit(X, target)
        pred_min = model.predict(X_eval)
        if scenario == 'S2':
            pred_min = np.expm1(pred_min)
        pred_min = np.clip(pred_min, 0, None)
        return model, to_class(pred_min), pred_min
    if scenario == 'S4':
        if algo == 'xgb':
            model.fit(X, y_cls, sample_weight=compute_sample_weight('balanced', y_cls))
        else:
            model.set_params(**{('logisticregression__' if algo == 'lr' else '') + 'class_weight': 'balanced'})
            model.fit(X, y_cls)
    else:
        model.fit(X, y_cls)
    return model, model.predict(X_eval).astype(int), None

## 6. Latih & evaluasi

In [ ]:
def class_metrics(y_true, y_pred):
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=CLASSES, zero_division=0)
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, labels=CLASSES, average='macro', zero_division=0),
        'weighted_f1': f1_score(y_true, y_pred, labels=CLASSES, average='weighted', zero_division=0),
        'recall_class_5': r[5], 'precision_class_5': p[5],
        'per_class': {str(c): {'precision': p[i], 'recall': r[i], 'f1': f[i], 'support': int(s[i])} for i, c in enumerate(CLASSES)},
        'confusion_matrix': confusion_matrix(y_true, y_pred, labels=CLASSES).tolist(),
    }

def cv_scores(approach, scenario, algo):
    idx = np.random.RandomState(SEED).choice(len(X_train), min(CV_N, len(X_train)), replace=False)
    Xc, ym, yc = X_train.iloc[idx], y_train_min[idx], y_train_cls[idx]
    f1s, r5s = [], []
    for tr, va in StratifiedKFold(5, shuffle=True, random_state=SEED).split(Xc, yc):
        _, pred, _ = fit_predict(approach, scenario, algo, Xc.iloc[tr], ym[tr], yc[tr], Xc.iloc[va])
        f1s.append(f1_score(yc[va], pred, labels=CLASSES, average='macro', zero_division=0))
        r5s.append(precision_recall_fscore_support(yc[va], pred, labels=[5], zero_division=0)[1][0])
    return {'cv_macro_f1_mean': float(np.mean(f1s)), 'cv_macro_f1_std': float(np.std(f1s)),
            'cv_recall_class_5_mean': float(np.mean(r5s)), 'cv_recall_class_5_std': float(np.std(r5s))}

def to_py(o):
    if isinstance(o, dict): return {k: to_py(v) for k, v in o.items()}
    if isinstance(o, list): return [to_py(v) for v in o]
    return o.item() if isinstance(o, np.generic) else o

for approach, scenario, algo in SPECS:
    name = f'{approach}_{scenario}_{algo}'
    path = os.path.join(RESULT_DIR, name + '.json')
    if os.path.exists(path):
        print(f'{name}: sudah ada, dilewati'); continue
    t = time.time()
    model, pred_cls, pred_min = fit_predict(approach, scenario, algo, X_train, y_train_min, y_train_cls, X_test)
    fit_s = time.time() - t
    metrics = class_metrics(y_test_cls, pred_cls)
    if pred_min is not None:
        metrics.update(mae=mean_absolute_error(y_test_min, pred_min),
                       rmse=float(np.sqrt(mean_squared_error(y_test_min, pred_min))),
                       r2=r2_score(y_test_min, pred_min))
    metrics.update(cv_scores(approach, scenario, algo))
    metrics['train_seconds'] = round(fit_s, 1)
    if algo == 'xgb':  # hanya XGBoost yang disimpan (untuk ekspor ONNX di 04_export)
        model.save_model(os.path.join(MODEL_DIR, f'xgb_{approach}_{scenario}.json'))
    with open(path, 'w') as f:
        json.dump(to_py({'approach': approach, 'scenario': scenario, 'algorithm': algo, 'metrics': metrics}), f)
    print(f"{name}: macro F1 {metrics['macro_f1']:.3f} | recall kelas 5 {metrics['recall_class_5']:.3f} | {time.time() - t:.0f} detik")

## 7. Rekap hasil

In [ ]:
results = []
for approach, scenario, algo in SPECS:
    with open(os.path.join(RESULT_DIR, f'{approach}_{scenario}_{algo}.json')) as f:
        results.append(json.load(f))

NAMES = {'lr': 'LR / LogReg', 'dt': 'Decision Tree', 'rf': 'Random Forest', 'xgb': 'XGBoost'}
table = pd.DataFrame([{
    'pendekatan': 'A Regresi' if r['approach'] == 'regression' else 'B Klasifikasi',
    'skenario': r['scenario'], 'algoritma': NAMES[r['algorithm']],
    'macro_f1': r['metrics']['macro_f1'], 'recall_k5': r['metrics']['recall_class_5'],
    'precision_k5': r['metrics']['precision_class_5'], 'accuracy': r['metrics']['accuracy'],
    'cv_macro_f1': f"{r['metrics']['cv_macro_f1_mean']:.3f} ± {r['metrics']['cv_macro_f1_std']:.3f}",
    'mae': r['metrics'].get('mae'), 'rmse': r['metrics'].get('rmse'), 'r2': r['metrics'].get('r2'),
    'detik': r['metrics']['train_seconds'],
} for r in results])
table.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
labels = table.pendekatan.str[0] + '-' + table.skenario + ' ' + table.algoritma
colors = [BLUE2 if p.startswith('A') else BLUE for p in table.pendekatan]
for ax, col, title in [(axes[0], 'macro_f1', 'Macro F1'), (axes[1], 'recall_k5', 'Recall kelas 5 (> 240 menit)')]:
    ax.barh(labels, table[col], color=colors if col == 'macro_f1' else ORANGE)
    ax.set(title=title, xlim=(0, max(0.05, table[col].max() * 1.25))); ax.invert_yaxis()
    for i, v in enumerate(table[col]):
        ax.text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=8)
fig.suptitle('Perbandingan 16 model (biru muda = A Regresi, biru tua = B Klasifikasi)', fontweight='bold')
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, '08_perbandingan_model.png'), dpi=200, bbox_inches='tight'); plt.show()

## 8. Model terbaik

Aturan pemilihan: **macro F1 tertinggi**, recall kelas 5 sebagai pembanding.
Model untuk **web** = XGBoost terbaik (ukuran ONNX kecil); kalau model terbaik keseluruhan bukan XGBoost, keduanya dilaporkan di paper.

In [ ]:
best = max(results, key=lambda r: (r['metrics']['macro_f1'], r['metrics']['recall_class_5']))
best_xgb = max((r for r in results if r['algorithm'] == 'xgb'),
               key=lambda r: (r['metrics']['macro_f1'], r['metrics']['recall_class_5']))
for label, r in [('Terbaik keseluruhan', best), ('XGBoost untuk web', best_xgb)]:
    print(f"{label}: {r['approach']} {r['scenario']} {r['algorithm']} — macro F1 {r['metrics']['macro_f1']:.3f}, recall kelas 5 {r['metrics']['recall_class_5']:.3f}")

cm = np.array(best['metrics']['confusion_matrix'])
cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm_pct, cmap='Blues', vmin=0, vmax=100)
for i in range(6):
    for j in range(6):
        ax.text(j, i, f'{cm_pct[i, j]:.0f}%', ha='center', va='center', fontsize=9,
                color='white' if cm_pct[i, j] > 50 else 'black')
ax.set(xticks=range(6), yticks=range(6), xlabel='Prediksi', ylabel='Aktual',
       title=f"Confusion matrix (% per kelas aktual)\n{best['approach']} {best['scenario']} {best['algorithm']}")
ax.grid(False); fig.colorbar(im, ax=ax)
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, '09_confusion_matrix.png'), dpi=200, bbox_inches='tight'); plt.show()

In [ ]:
xgb_model = (XGBRegressor() if best_xgb['approach'] == 'regression' else XGBClassifier())
xgb_model.load_model(os.path.join(MODEL_DIR, f"xgb_{best_xgb['approach']}_{best_xgb['scenario']}.json"))
imp = pd.Series(xgb_model.get_booster().get_score(importance_type='gain'))
imp.index = [FEATURES[int(k[1:])] if k.startswith('f') and k[1:].isdigit() else k for k in imp.index]
imp = (imp / imp.sum() * 100).sort_values()
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(imp.index, imp.values, color=BLUE2)
ax.set(title='Feature importance XGBoost (gain, %)', xlabel='% kontribusi')
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, '10_feature_importance.png'), dpi=200, bbox_inches='tight'); plt.show()

with open(os.path.join(MODEL_DIR, 'best.json'), 'w') as f:
    json.dump({'best_overall': {k: best[k] for k in ('approach', 'scenario', 'algorithm')},
               'best_xgb': {k: best_xgb[k] for k in ('approach', 'scenario', 'algorithm')},
               'n_sample': int(len(df)), 'n_train': int(len(X_train)), 'n_test': int(len(X_test))}, f, indent=2)

## 9. Catatan untuk paper

- Bandingkan A vs B pada **macro F1** dan **recall kelas 5** → menjawab H1–H3.
- Efek S2 (log) terhadap regresi dan S4 (class weight) terhadap klasifikasi.
- Akurasi tinggi tapi recall kelas 5 rendah = bukti akurasi menyesatkan pada data timpang.
- Keterbatasan: tanpa tuning hyperparameter, sampel 1 juta baris, fitur tanpa informasi hari-H (delay penerbangan sebelumnya, cuaca).